<a href="https://colab.research.google.com/github/claudiovaliense/emnlp-2026-Beyond-Global-Calibration-/blob/main/Beyond_Global_Calibration_Analyzing_Class_Specific_Miscalibration_and_High_Confidence_Errors_in_Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install netcal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 27.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [netcal]


In [ ]:
from netcal.metrics import ECE
import numpy as np

def ece_claudio(list_prob, list_label):
  ece = ECE(bins=10)
  return ece.measure( np.array( list_prob), np.array( list_label ) )

def ece_classwise(prob, y_true, n_bins=10):

    prob = np.asarray(prob)
    y_true = np.asarray(y_true)

    n_classes = prob.shape[1]

    class_ece = []

    for k in range(n_classes):

        # Probabilidade atribuída à classe k
        confidence = prob[:, k]

        # 1 se k é a classe verdadeira
        correct = (y_true == k).astype(int)

        bin_edges = np.linspace(0, 1, n_bins + 1)

        ece = 0.0

        for i in range(n_bins):

            if i == n_bins - 1:
                mask = (
                    (confidence >= bin_edges[i]) &
                    (confidence <= bin_edges[i + 1])
                )
            else:
                mask = (
                    (confidence >= bin_edges[i]) &
                    (confidence < bin_edges[i + 1])
                )

            if np.sum(mask) == 0:
                continue

            accuracy = np.mean(correct[mask])
            confidence_mean = np.mean(confidence[mask])

            weight = np.sum(mask) / len(confidence)

            ece += weight * abs(
                accuracy - confidence_mean
            )

        class_ece.append(ece)

    return np.array(class_ece)

import numpy as np
from sklearn.preprocessing import OneHotEncoder

def brier_multi(list_label, list_prob):
    """print(1- brier_multi( df.y_true, df.prob ) )"""
    one = OneHotEncoder()
    list_label = np.array(list_label)
    list_prob = np.array(list_prob)
    list_label = np.array(  one.fit_transform( list_label .reshape(-1, 1)  ).todense() )

    return np.mean(np.sum((list_prob - list_label)**2, axis=1)) # aqui fica o intervalo de 0 a 2, implementacao original
    # return np.mean(np.sum((list_prob - list_label)**2, axis=1)) /2  # aqui o intervalo fica de 0 a 1, aqui fica similar ao scikit

import numpy as np
from sklearn.linear_model import LogisticRegression

def calibracao_platt_multiclasse2(prob_treinar,label_treinar,prob_aplicar):

    probs_train = np.array(prob_treinar, dtype=float)
    y_train = np.array(label_treinar)

    probs_test = np.array( prob_aplicar , dtype=float)

    n_classes = probs_train.shape[1]

    calibrated_test = np.zeros_like(probs_test)

    for k in range(n_classes):

        # One-vs-Rest
        y_bin = (y_train == k).astype(int)

        lr = LogisticRegression(solver="lbfgs")

        #aproximanao menos proxima do original logit----
        lr.fit(
            probs_train[:, k].reshape(-1, 1),
            y_bin
        )
        calibrated_test[:, k] = lr.predict_proba(
            probs_test[:, k].reshape(-1, 1)
        )[:, 1]
        #aproximanao menos proxima---------

        # #aplica llog-odds ficar mais proximo do  logits, que é necessario para usar platt de forma original
        # eps = 1e-15
        # x_train = np.log(np.clip(probs_train[:, k], eps, 1-eps) /
        #                  (1 - np.clip(probs_train[:, k], eps, 1-eps)))
        # x_test = np.log(np.clip(probs_test[:, k], eps, 1-eps) /
        #                 (1 - np.clip(probs_test[:, k], eps, 1-eps)))
        # lr.fit(x_train.reshape(-1, 1), y_bin)
        # calibrated_test[:, k] = lr.predict_proba(
        #     x_test.reshape(-1, 1)
        # )[:, 1]
        # #aplica llog-odds ------

    # Renormaliza para que cada linha some 1
    row_sum = calibrated_test.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0

    calibrated_test /= row_sum

    return calibrated_test.tolist(), calibrated_test.max(axis=1)

from sklearn.isotonic import IsotonicRegression
import numpy as np

def calibracao_isotonic_divisao_validacao_multiclassse2(prob_trenar, label_treinar, prob_aplicar):
    """
    probs: np.array shape (n_samples, n_classes)
    y_true: np.array shape (n_samples,)
    """
    probs_train = np.array( prob_trenar )
    y_train = np.array( label_treinar )
    probs_test = np.array( prob_aplicar )

    n_classes = np.array( prob_trenar ).shape[1]
    calibrated_test = np.zeros_like( np.array( prob_aplicar ) )

    for k in range(n_classes):
        ir = IsotonicRegression(out_of_bounds='clip')
        # target binária: 1 se a classe verdadeira é k
        y_bin = ( y_train == k).astype(int)
        ir.fit(probs_train[:, k], y_bin)
        calibrated_test[:, k] = ir.transform( probs_test [:, k])


    # normalizar para manter soma = 1
    calibrated_test = calibrated_test / calibrated_test.sum(axis=1, keepdims=True)
    return list(calibrated_test), np.max(calibrated_test, axis=1)

In [ ]:
from re import L
import pandas as pd
import numpy as np

df = pd.read_parquet('/content/sst_roberta_fold_0.parquet')
df


,idx,text,label,prob,pred
0,0,The Rock is destined to be the 21st Century 's...,1,"[0.0017700569005683064, 0.9982299208641052]",1
1,1,The gorgeously elaborate continuation of `` Th...,1,"[0.0022921827621757984, 0.9977078437805176]",1
2,2,Singer\/composer Bryan Adams contributes a sle...,1,"[0.009241325780749321, 0.9907586574554443]",1
3,3,You 'd think by now America would have had eno...,0,"[0.9943190217018127, 0.005680986680090427]",0
4,4,Yet the act is still charming here .,1,"[0.0009510783129371703, 0.9990488886833191]",1
...,...,...,...,...,...
11836,11836,An imaginative comedy\/thriller .,1,"[0.0011932308552786708, 0.9988067150115967]",1
11837,11837,"( A ) rare , beautiful film .",1,"[0.0009675259934738278, 0.9990324974060059]",1
11838,11838,( An ) hilarious romantic comedy .,1,"[0.0012935601407662034, 0.9987064599990845]",1
11839,11839,Never ( sinks ) into exploitation .,1,"[0.2536078691482544, 0.7463921308517456]",1


In [ ]:
#examples
df_train = df.sample(9000)
df_test =  df.drop(df_train.index).sample(1000, random_state=42)

print('ECE')
print( ece_claudio( list( df_test['prob']), list(df_test['label'] ) )  )
print('ECE by class')
print( ece_classwise( list( df_test['prob']), list(df_test['label'] ) )  )
print('Brier Score')
print( brier_multi( list(df_test['label']), list(df_test['prob'])  )  )

print('\nCALIBRATION-------- BRIER SCORE\n')
print('PLATT calibration')
prob_calibrado_platt, confianca_max_platt = calibracao_platt_multiclasse2(list(df_train.prob), list(df_train.label), list(df_test.prob))
print( brier_multi( list(df_test['label']), list(prob_calibrado_platt)  )  )

print('Isotonic calibration')
prob_calibrado_iso, confianca_max_iso = calibracao_isotonic_divisao_validacao_multiclassse2(list(df_train.prob), list(df_train.label), list(df_test.prob))
print( brier_multi( list(df_test['label']), list(prob_calibrado_iso)  )  )

ECE
0.03379264832730407
ECE by class
[0.03379265 0.03379265]
Brier Score
0.08214226651051865

CALIBRATION-------- BRIER SCORE

PLATT calibration
0.08105895979042406
Isotonic calibration
0.07950927242412172
